# Chroma DB

In [ ]:
# !pip install chromadb

In [7]:
import chromadb

chroma_client = chromadb.Client()

In [8]:
collection = chroma_client.create_collection(name="my_collection")

In [10]:
collection.add(
    documents=[
        "This is a document about pineapples",
        "This is a document about mango",
        "This is a document about strawberry"
    ],
    ids=['id1', 'id2', 'id3']
)

C:\Users\Playdata\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.39MiB/s]


In [13]:
results = collection.query(
    query_texts=["This is a query document about vietnam"],
    n_results=3

)

In [14]:
results

{'ids': [['id1', 'id2', 'id3']],
 'embeddings': None,
 'documents': [['This is a document about pineapples',
   'This is a document about mango',
   'This is a document about strawberry']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[1.233154296875, 1.2783520221710205, 1.3223111629486084]]}

### SciQ dataset 활용 ChromaDB 검색

In [16]:
# 데이터셋 로드
from datasets import load_dataset

dataset = load_dataset("sciq", split="train")
dataset = dataset.filter(lambda x: x["support"] != "")

dataset

Filter: 100%|██████████| 11679/11679 [00:00<00:00, 87149.08 examples/s]


Dataset({
    features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
    num_rows: 10481
})

In [18]:
# chroma db 클라이언트 객체 및 콜렉션 생성

import chromadb

client = chromadb.Client()
collection = client.create_collection(name="sciq_support")

In [19]:
# 임베딩 모델 로드
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\Playdata\anaconda3\envs\vectordb_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. 

In [20]:
supports = dataset["support"][:100]
supports_embeddings = embedding_model.encode(supports).tolist()

In [21]:
len(supports_embeddings[0])

384

In [24]:
collection.add(
    ids=[str(i) for i in range(0, 100)],
    embeddings=supports_embeddings,
    metadatas=[{"type": "support", "text": text} for text in supports]
)

In [26]:
questions = dataset['question'][:3]
question_embeddings = embedding_model.encode(questions).tolist()

results = collection.query(
    query_embeddings=question_embeddings,
    n_results=1
)

In [27]:
results

{'ids': [['36'], ['1'], ['2']],
 'embeddings': None,
 'documents': [[None], [None], [None]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'text': 'Agents of Decomposition The fungus-like protist saprobes are specialized to absorb nutrients from nonliving organic matter, such as dead organisms or their wastes. For instance, many types of oomycetes grow on dead animals or algae. Saprobic protists have the essential function of returning inorganic nutrients to the soil and water. This process allows for new plant growth, which in turn generates sustenance for other organisms along the food chain. Indeed, without saprobe species, such as protists, fungi, and bacteria, life would cease to exist as all organic carbon became “tied up” in dead organisms.',
    'type': 'support'}],
  [{'text': 'Without Coriolis Effect the global winds would blow north to south or south to north. But Coriolis makes them blow northeast to southwest or the re

In [28]:
for i, q in enumerate(questions):
    print(f"Question: {q}")
    print(f"Support: {results['metadatas'][i][0]['text']}")
    print("-"*100)


Question: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Support: Agents of Decomposition The fungus-like protist saprobes are specialized to absorb nutrients from nonliving organic matter, such as dead organisms or their wastes. For instance, many types of oomycetes grow on dead animals or algae. Saprobic protists have the essential function of returning inorganic nutrients to the soil and water. This process allows for new plant growth, which in turn generates sustenance for other organisms along the food chain. Indeed, without saprobe species, such as protists, fungi, and bacteria, life would cease to exist as all organic carbon became “tied up” in dead organisms.
----------------------------------------------------------------------------------------------------
Question: What phenomenon makes global winds blow northeast to southwest or the reverse in the northern hemisphere and northwest to southeast or the reverse in the southern hemisph

### Chroma DB를 활용한 키워드 기반 검색

In [29]:
documents = [
    "인공지능은 인간의 작업을 자동화하는 기술이다.",
    "기계 학습은 데이터에서 패턴을 학습하여 예측하는 기술이다.",
    "벡터 데이터베이스는 유사도를 기반으로 데이터를 검색하는 DB이다.",
    "임베딩은 텍스트나 이미지를 수치 벡터로 변환하는 과정이다.",
    "크로마DB는 벡터 임베딩을 효율적으로 저장하고 검색할 수 있는 데이터베이스이다.",
    "유사도 검색은 쿼리 벡터와 가장 가까운 벡터를 찾는 과정을 의미한다.",
    "자연어 처리는 인간의 언어를 이해하고 생성하는 인공지능 기술이다.",
    "RAG(Retrieval-Augmented Generation)는 검색과 생성 모델을 결합하여 정확한 답변을 생성하는 방법이다.",
    "크로마DB는 오픈소스 벡터 데이터베이스로, 로컬 환경에서도 쉽게 사용할 수 있다.",
    "임베딩 모델은 단어, 문장, 문서를 벡터 공간에 매핑하여 의미적 유사도를 계산할 수 있게 한다.",
    "대규모 언어 모델은 방대한 텍스트 데이터를 학습하여 다양한 자연어 작업을 수행할 수 있다.",
    "크로마DB의 컬렉션(collection)은 서로 다른 데이터 그룹을 분리하여 관리할 수 있게 한다.",
    "메타데이터는 문서의 부가 정보를 저장하여 검색 결과를 필터링할 때 유용하게 사용된다.",
    "크로마DB는 파이썬 환경에서 간단한 코드로 벡터를 추가하고 검색할 수 있다.",
    "효율적인 검색을 위해 벡터 차원을 임베딩 모델에 맞게 설정하는 것이 중요하다.",
    "RAG 시스템은 외부 지식베이스에서 관련 정보를 검색하여 응답의 신뢰도를 높인다.",
    "벡터 검색은 키워드 기반 검색보다 의미적으로 더 유사한 결과를 반환한다.",
    "크로마DB는 OpenAI 임베딩 모델과 함께 자주 사용된다.",
    "데이터 전처리는 임베딩 품질을 높이기 위해 불필요한 기호나 중복 문장을 제거하는 과정이다.",
    "하이브리드 검색은 벡터 검색과 키워드 검색을 결합하여 검색 성능을 향상시킨다.",
    "크로마DB는 메모리 내(in-memory) 또는 디스크 기반으로 데이터를 저장할 수 있다.",
    "벡터 인덱싱은 대규모 데이터에서도 빠른 유사도 계산을 가능하게 한다.",
    "RAG 파이프라인은 검색 단계와 생성 단계를 순차적으로 수행하여 더 풍부한 결과를 생성한다."
]

In [30]:
# ChromaDB 클라이언트, 컬렉션 생성
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name='ai_documents')

In [31]:
# 1. 임베딩 모델 로드
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. 임베딩 처리 → add() 메서드 활용
for i, doc in enumerate(documents):
    embedding = embedding_model.encode(doc).tolist()
    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        metadatas=[{"text": doc}]
    )

In [39]:
query_keyword = "RAG"

query_embedding = embedding_model.encode(query_keyword).tolist()

results = collection.query(query_embeddings=query_embedding, n_results=3)

for result in results["metadatas"][0]:
    print("검색된 문서: ", result["text"])

검색된 문서:  RAG 시스템은 외부 지식베이스에서 관련 정보를 검색하여 응답의 신뢰도를 높인다.
검색된 문서:  RAG(Retrieval-Augmented Generation)는 검색과 생성 모델을 결합하여 정확한 답변을 생성하는 방법이다.
검색된 문서:  RAG 파이프라인은 검색 단계와 생성 단계를 순차적으로 수행하여 더 풍부한 결과를 생성한다.


**과제**

### 영화 추천 시스템

In [41]:
import pandas as pd

df = pd.read_csv("./data/tmdb_5000_movies.csv")
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [42]:
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [ ]:
# 1. 제목 입력 → 줄거리를 찾아 → 줄거리로 유사도 검색 → 유사도가 높은 영화 추천
# 사용자로 부터 입력 받는 것 (title) → overview 찾기 → overview 임베딩 → vectordb 검색 → 유사도가 높은 영화 추천
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="movies")

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


movies = [
    {
        "id": str(i),
        "title": row["title"],
        "overview": row["overview"] if pd.notna(row["overview"]) else ""
    }
    for i, row in df.iterrows()
]

for movie in movies:
    if movie['overview']:
        overview_embedding = embedding_model.encode(movie['overview']).tolist()
        collection.add(
            ids=[movie['id']],
            embeddings=[overview_embedding],
            metadatas=[{"title": movie['title'], "overview": movie['overview']}]
        )

"""
for i, row in df.iterrows():
    overview = row["overview"]
    title = row["title"]
    embedding = embedding_model.encode(overview).tolist()
    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        metadatas=[{"title": title, "overview": overview}]
    )
"""

'\nfor i, row in df.iterrows():\n    overview = row["overview"]\n    title = row["title"]\n    embedding = embedding_model.encode(overview).tolist()\n    collection.add(\n        ids=[str(i)],\n        embeddings=[embedding],\n        metadatas=[{"title": title, "overview": overview}]\n    )\n'

In [57]:
# 1. 제목 입력 → 줄거리를 찾아 → 줄거리로 유사도 검색 → 유사도가 높은 영화 추천

input_title = 'Inception'
query_text = df.loc[df['title'] == input_title, 'overview'].iloc[0]

query_embedding = embedding_model.encode(query_text).tolist()

results = collection.query(query_embeddings=[query_embedding], n_results=5)

for result in results['metadatas'][0]:
    print(result['title'])
    print(result['overview'])
    print()

Inception
Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person's idea into a target's subconscious.

Identity Thief
When a mild-mannered businessman learns his identity has been stolen, he hits the road in an attempt to foil the thief -- a trip that puts him in the path of a deceptively harmless-looking woman.

The Master of Disguise
A sweet-natured Italian waiter named Pistachio Disguisey at his father Fabbrizio's restaurant, who happens to be a member of a family with supernatural skills of disguise. But moments later the patriarch of the Disguisey family is kidnapped Fabbrizio's former arch-enemy, Devlin Bowman, a criminal mastermind in an attempt to steal the world's most precious treasures from around the world. And it's up to Pistachio to track down Bowman and save his family before Bowman ki

In [59]:
# 2. 원하는 줄거리 입력 → 유사도 검색

query_text = 'Korea'

query_embedding = embedding_model.encode(query_text).tolist()

results = collection.query(query_embeddings=[query_embedding], n_results=5)

for result in results['metadatas'][0]:
    print(result['title'])
    print(result['overview'])
    print()

Silmido
On 31 January 1968, 31 North Korean commandos infiltrated South Korea in a failed mission to assassinate President Park Chung-hee. In revenge, the South Korean military assembled a team of 31 criminals on the island of Silmido to kill Kim Il-sung for a suicide mission to redeem their honor, but was cancelled, leaving them frustrated. It is loosely based on a military uprising in the 1970s.

Tae Guk Gi: The Brotherhood of War
In 1950, in South Korea, shoe-shiner Jin-tae Lee and his 18-year-old old student brother, Jin-seok Lee, form a poor but happy family with their mother, Jin-tae's fiancé Young-shin Kim, and her young sisters. Jin-tae and his mother are tough workers, who sacrifice themselves to send Jin-seok to the university. When North Korea invades the South, the family escapes to a relative's house in the country, but along their journey, Jin-seok is forced to join the army to fight in the front, and Jin-tae enlists too to protect his young brother. The commander promise

### 논문 pdf 내용 검색

In [60]:
!pip install PyPDF2

In [77]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path="./chroma_db")
# client.delete_collection(name="papers") # collection 삭제
collection = client.get_or_create_collection(name='papers')

model = SentenceTransformer('all-MiniLM-L6-v2')

In [78]:
papers = [
    {'id':1, 'title': '딥러닝', 'path': './data/deep_learning.pdf'},
    {'id':2, 'title': '자연어 처리', 'path': './data/nlp_paper.pdf'}
]

In [79]:
import PyPDF2

def extract_text_from_pdf(path):
    with open(path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        text = " ".join([page.extract_text() for page in reader.pages if page.extract_text()])
    return text

In [80]:
for paper in papers:
    text = extract_text_from_pdf(paper['path'])
    embedding = model.encode(text).tolist()
    collection.add(
        ids=[paper['id']],
        embeddings=[embedding],
        metadatas=[{'title': paper['title']}],
        documents=[text]
    )

ValueError: Expected ID to be a str, got 1 in add.

In [ ]:
collection.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [ ]:
results = collection.get(include=['embeddings', 'documents', 'metadatas'])

for emb in results['embeddings']:
    print(len(emb))

In [ ]:
query_text = 'Natural Language'

query_embedding = model.encode(query_text).tolist()
results = collection.query(query_embeddings=[query_embedding], n_results=1)

results['metadatas'][0][0]['title']